# What's New in gVXR 2.1.1

This tutorial introduces the major new features added in **gVXR 2.1.1**. It is intended for users who are already familiar with the basics of gVXR, including loading geometries, defining X-ray sources and detectors, and computing radiographs.

## Learning objectives

By the end of this tutorial, you will be able to:

1. Navigate and inspect a scene using the new camera system.
2. Interactively adjust acquisition geometry using the mouse and keyboard.
3. Load scanner parameters directly from an XTekCT configuration file.
4. Check whether points lie within an object's bounding box.
5. Compute and export object silhouettes.
6. Use `gVXRDataReader` with both file-based and in-memory data.
7. Understand recent API changes and deprecated functionality.
8. Take advantage of improved visualisation and interaction features.

---

## Table of Contents

1. [Navigating a Scene with the New Camera System](#1-navigating-a-scene-with-the-new-camera-system)
2. [Rotating the Sample Interactively](#2-rotating-the-sample-interactively)
3. [Interactively Adjusting Imaging Geometry](#3-interactively-adjusting-imaging-geometry)
4. [Panning the View](#4-panning-the-view)
5. [Built-in Help](#5-built-in-help)
6. [Copying Screenshots to the Clipboard](#6-copying-screenshots-to-the-clipboard)
7. [Importing XTekCT Geometries](#7-importing-xtekct-geometries)
8. [Bounding-Box Queries](#8-bounding-box-queries)
9. [Computing Silhouettes](#9-computing-silhouettes)
10. [Reading Data from Memory](#10-reading-data-from-memory)
11. [Projection Matrices with Explicit Units](#11-projection-matrices-with-explicit-units)
12. [Visualisation Improvements](#12-visualisation-improvements)
13. [Migration Notes](#migration-notes)
14. [Capstone Exercise](#capstone-exercise)

---

# 1. Navigating a Scene with the New Camera System

## Overview

gVXR now provides a dedicated camera system that supports:

- Rotating around the scene;
- Panning the view;
- Zooming in and out;
- Direct manipulation of camera parameters.

This behaviour is similar to modern CAD and scientific visualisation software and makes scene exploration considerably more intuitive.

## Exercise

### Objective

Explore the new camera system and become familiar with scene navigation.

### Steps

1. Load a sample.
2. Start the interactive viewer:

   ```python
   gvxr.renderLoop()
   ```

or
3. Use the `ViewerPanel` Jupyter Widget.

```python
from gvxrPython3.notebook import ViewerPanel
...
ViewerPanel()
```

4. Rotate around the sample.
5. Zoom in on a small feature.
6. Pan across the scene.
7. Inspect the sample from different viewpoints.

### Expected output

You should be able to:

- Rotate the camera around the sample;
- Zoom smoothly towards and away from the sample;
- Pan the view without moving the sample;
- Inspect the sample from arbitrary viewpoints.

### Questions

- What is the difference between moving the camera and rotating the sample?
- Which approach is preferable when inspecting a complex assembly?
- How does camera navigation simplify scene exploration?

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

gvxr.renderLoop()
gvxr.destroy()

---

# 2. Rotating the Sample Interactively

## Overview

The sample can now be rotated directly using the middle mouse button and mouse movement.

This allows users to quickly investigate how object orientation affects projections.

## Exercise

### Objective

Investigate the effect of sample orientation on generated projections.

### Steps

1. Load a mesh:

   ```python
   gvxr.loadMeshFile(...)
   ```

2. Open the interactive viewer.
3. Rotate the sample using the middle mouse button.
4. Observe how the projected shape changes.
5. Compare several viewing angles.

### Expected output

You should observe:

- A smooth rotation of the object in the viewer;
- Changes in the object's orientation relative to the beam;
- Corresponding changes in the projected silhouette and radiograph.

### Questions

- Which orientations produce the largest projected area?
- Which features become hidden in certain views?
- How would object orientation affect a tomography acquisition?

### Challenge (optional)

Try the same experiment with a CAD assembly containing multiple components and identify orientations that produce overlapping structures.

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

print("Use the middle mouse button to rotate the sample.")

gvxr.renderLoop()
gvxr.destroy()

# 3. Interactively Adjusting Imaging Geometry

## Overview

The source-to-object distance (SOD) and object-to-detector distance (ODD) can now be adjusted interactively using the keyboard and mouse.

Collision detection prevents invalid configurations.

## Exercise

### Objective

Investigate the effect of geometry changes on magnification.

### Steps

1. Load a sample.
2. Increase and decrease the SOD using:

   ```text
   S + ↑
   S + ↓
   ```

3. Increase and decrease the ODD using:

   ```text
   D + ↑
   D + ↓
   ```

4. Repeat using the mouse wheel.
5. Repeat using mouse dragging.
6. Observe the projection size on the detector.

### Expected output

You should observe:

- An increase in magnification as the source approaches the sample;
- Changes in projected object size as the detector position changes;
- Automatic prevention of source or detector collisions with the sample.

### Questions

- How does magnification change as the source approaches the sample?
- How does detector position affect the field of view?
- When does collision detection prevent further movement?

### Challenge (optional)

Determine a geometry that maximises magnification without clipping the object from the detector.

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

print("Adjust geometry interactively:")
print("  S + arrows / mouse = source")
print("  D + arrows / mouse = detector")

gvxr.renderLoop()
gvxr.destroy()

# 4. Panning the View

## Overview

The camera can now be translated laterally, making it easier to inspect large scenes and off-centre objects.

## Exercise

### Objective

Use camera panning to explore a scene.

### Steps

1. Load a scene containing several objects.
2. Zoom in on one object.
3. Pan the camera across the scene.
4. Inspect every object without moving the sample itself.

### Expected output

You should be able to:

- Move the viewpoint horizontally and vertically;
- Inspect different areas of the scene;
- Keep the sample position unchanged while exploring the environment.

### Questions

- When is panning preferable to zooming?
- Does panning help inspect large assemblies?
- How does panning improve scene navigation?

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

print("Use the mouse to pan across the scene.")

gvxr.renderLoop()
gvxr.destroy()

# 5. Built-in Help

## Overview

A help screen is now available directly from the interactive viewer.

## Exercise

### Objective

Become familiar with all available keyboard and mouse controls.

### Steps

1. Open the interactive viewer.
2. Display the help screen.
3. Review all available controls.
4. Experiment with the navigation shortcuts.
5. Experiment with the geometry-manipulation shortcuts.

### Expected output

You should see:

- A list of keyboard shortcuts;
- A list of mouse controls;
- Descriptions of available interactions.

### Questions

- Which controls are most useful for your workflow?
- Which controls were not available in previous releases?

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

print("Press H to hide/show the help message.")

gvxr.renderLoop()
gvxr.destroy()

# 6. Copying Screenshots to the Clipboard

## Overview

The latest screenshot can now be copied directly to the system clipboard.

This simplifies creating presentations, documentation, tutorials, and bug reports.

## Exercise

### Objective

Capture and reuse screenshots produced by gVXR.

### Steps

1. Generate a projection:

   ```python
   image = gvxr.computeXRayImage()
   ```

2. Capture a screenshot.
3. Copy it to the clipboard.
4. Paste it into a presentation, document, or e-mail.

### Expected output

You should obtain:

- A screenshot copied to the system clipboard;
- A successful paste operation in another application;
- A faithful reproduction of the image generated by gVXR.

### Questions

- Does the image appear identical after pasting?
- How could this feature simplify bug reporting or teaching activities?

### Challenge (optional)

Create a short slide deck illustrating different acquisition geometries using clipboard screenshots.

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

print("Capture a screenshot from the viewer.")
print("Copy it to the clipboard using the viewer shortcut.")
print("Paste it into a slideshow for a presentation.")

gvxr.renderLoop()
gvxr.destroy()

# 7. Importing XTekCT Geometries

## Overview

Acquisition parameters can now be imported directly from an `.xtekct` file.

## Exercise

### Objective

Configure an acquisition geometry from an XTekCT configuration file.

### Steps

1. Obtain an `.xtekct` file.
2. Import the acquisition parameters into gVXR.
3. Inspect the generated geometry.
4. Compare it with a manually configured setup.

### Expected output

You should obtain:

- A fully configured acquisition geometry;
- Correct source and detector positions;
- Matching geometry between the imported and manually configured setups.

### Questions

- Which parameters are imported automatically?
- Are the imported and manual geometries identical?
- How much setup time can be saved?

### Challenge (optional)

Import several XTekCT files from different scanners and compare their geometries.

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.loadXtekct("scan.xtekct")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

gvxr.computeXRayImage();
gvxr.displayScene()

gvxr.renderLoop()
gvxr.destroy()

# 8. Bounding-Box Queries

## Overview

The function `isPointInNodeAndChildrenBoundingBox()` determines whether a point is inside the bounding box of an object or any of its descendants.

## Exercise

### Objective

Use bounding-box queries to validate points in a scene.

### Steps

1. Load a sample.
2. Define several test coordinates.
3. Evaluate each coordinate using:

   ```cpp
   isPointInNodeAndChildrenBoundingBox(...)
   ```

4. Record the results.

### Expected output

You should obtain:

- A set of coordinates classified as inside or outside the bounding box;
- Consistent results for all tested points.

### Questions

- Which points are inside the bounding box?
- Which points are outside?
- How could this functionality support collision detection?

<!--### Challenge (optional)

Implement a simple picking tool that reports whether the mouse cursor intersects a bounding box.-->

## Reference Implementation

In [ ]:
import os
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

print("Bounding box:", 
      gvxr.getNodeAndChildrenBoundingBox("sample", "mm"), 
      "mm")

inside = gvxr.isPointInNodeAndChildrenBoundingBox(
    "sample",
    0.0, 0.0, 0.0, "mm"
)
print("Inside bounding box:", inside)

inside = gvxr.isPointInNodeAndChildrenBoundingBox(
    "sample",
    10.0, 0.0, 0.0, "mm"
)
print("Inside bounding box:", inside)

gvxr.destroy()

# 9. Computing Silhouettes

## Overview

Silhouettes provide a binary representation of object occupancy and are significantly smaller than L-buffer images.

## Exercise

### Objective

Generate and save a silhouette image.

### Steps

1. Compute the silhouette:

   ```python
   gvxr.computeSilhouette("sample")
   ```

2. Retrieve it:

   ```python
   silhouette = gvxr.getLastSilhouette()
   ```

3. Save it:

   ```python
   gvxr.saveLastSilhouette("silhouette.png")
   ```

4. Visualise the resulting image.

### Expected output

A binary image where:

- Background pixels have value 0;
- Object pixels have value 255;
- The object's projected shape is clearly visible;
- The silhouette file is saved successfully.

### Questions

- Which regions correspond to the object?
- Which regions correspond to the background?
- How does the silhouette differ from an X-ray projection?

### Challenge (optional)

Generate silhouettes from many viewing directions and build a dataset for machine-learning applications such as classification, detection, or pose estimation.

## Reference Implementation


In [ ]:
import os
from tifffile import imwrite
import matplotlib.pyplot as plt # Plotting
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

# Compute the silhouette:
silhouette1 = gvxr.computeSilhouette("sample")

# Retrieve it:
silhouette2 = gvxr.getLastSilhouette()

# Save it:
imwrite(silhouette1, "silhouette1.png")
gvxr.saveLastSilhouette("silhouette2.png")

# Visualise the resulting image.
plt.figure()
plt.subplot(121)
plt.imshow(silhouette1)
plt.subplot(122)
plt.imshow(silhouette2)
plt.show()

gvxr.destroy()

## Additional Exercise

### Objective

Compare silhouettes and L-buffer images.

### Steps

1. Compute an L-buffer:

   ```python
   gvxr.computeLBuffer()
   ```

2. Compute a silhouette:

   ```python
   gvxr.computeSilhouette()
   ```

3. Compare memory usage.
4. Compare visual appearance.
5. Compare suitability for segmentation tasks.

### Expected output

You should observe:

- A floating-point image for the L-buffer;
- A binary image for the silhouette;
- Smaller memory usage for the silhouette;
- Easier segmentation of the silhouette image.

### Questions

- Which representation is more compact?
- Which representation contains more physical information?
- Which representation is more suitable for machine-learning pipelines?

## Reference Implementation


In [ ]:
import os
import matplotlib.pyplot as plt # Plotting
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

path = os.path.dirname(gvxr.__file__)
fname = path + "/welsh-dragon-small.stl"
gvxr.loadMeshFile("Dragon", fname, "mm")
gvxr.moveToCentre("Dragon")
gvxr.setMixture("Dragon", ["Ti", "Al", "V"], [0.9, 0.06, 0.04])
gvxr.setDensity("Dragon", 4.43, "g/cm3")

# Compute and retrieve the L-buffer and the silhouette:
lbuffer = gvxr.computeLBuffer("sample")
silhouette = gvxr.computeSilhouette("sample")

# Visualise the resulting image.
plt.figure()
plt.subplot(121)
plt.title("L-buffer")
plt.imshow(lbuffer)
plt.subplot(122)
plt.title("Silhouette")
plt.imshow(silhouette)
plt.show()

print(sizeof())
print(sizeof())

gvxr.destroy()

Overlap of silhouette, binary mask
cube with sphere inside

## Reference Implementation

# 10. Reading Data from Memory

## Overview

`gVXRDataReader` can now work with data stored directly in memory.

## Exercise

### Objective

Compare disk-based and memory-based workflows.

### Steps

1. Create a volume in memory.
2. Load it using `gVXRDataReader`.
3. Load the same volume from disk.
4. Measure execution times.
5. Compare both workflows.

### Expected output

You should obtain:

- Successful loading from memory;
- Successful loading from disk;
- Timing measurements for both approaches.

### Questions

- Which workflow is faster?
- Is disk I/O a bottleneck?
- Which applications benefit most from in-memory data?

### Challenge (optional)

Create a workflow in which a reconstruction is generated and processed entirely in memory without writing intermediate files.

---


# 11. Projection Matrices with Explicit Units

## Overview

Projection matrix functions now allow explicit length units to be specified.

This reduces ambiguity when mixing datasets that use different units.

## Exercise

### Objective

Verify that equivalent lengths expressed with different units produce identical geometries.

### Steps

1. Create a projection matrix using:

   ```text
   100 mm
   ```

2. Create another using:

   ```text
   10 cm
   ```

3. Create another using:

   ```text
   0.1 m
   ```

4. Compare the resulting geometries.

### Expected output

You should observe:

- Identical viewing geometries;
- Matching field of view;
- Consistent projection behaviour regardless of chosen unit.

### Questions

- Are the geometries identical?
- Which workflow is least prone to unit-conversion errors?

---


# 12. Visualisation Improvements

## Overview

Zoom behaviour is now distance-aware and screenshots are generated in UINT8 format.

## Exercise

### Objective

Evaluate the new visualisation behaviour.

### Steps

1. Load a small object.
2. Zoom in and out repeatedly.
3. Repeat with a much larger object.
4. Capture screenshots.
5. Compare image quality and navigation responsiveness.

### Expected output

You should observe:

- Smooth and predictable zoom behaviour;
- Consistent navigation across different scene scales;
- Screenshots suitable for reports, documentation, and presentations.

### Questions

- Does zooming feel more natural?
- Is navigation consistent across differently sized datasets?
- Are the screenshots publication-ready?

### Challenge (optional)

Prepare a one-page figure showing the same object at several zoom levels and acquisition geometries using only screenshots generated by gVXR.

---


# 13. Migration Notes

The following functions have been removed:

```cpp
void setSceneRotationMatrix(const std::vector<double>& aRotationMatrix);
std::vector<double> getSceneRotationMatrix();

void setZoom(float aZoomValue);
float getZoom();
```

## Recommended Replacement

Use the new camera system instead of directly manipulating:

- Scene rotation matrices;
- Arbitrary zoom values.

The new approach is more intuitive, more robust, and better aligned with modern visualisation workflows.

---


# 14. Capstone Exercise

## Objective

Combine the new features introduced in gVXR 2.1.1 into a complete workflow.

### Steps

1. Import an XTekCT geometry.
2. Load a CAD model.
3. Explore the scene using the camera controls.
4. Adjust SOD and ODD interactively.
5. Validate positions using bounding-box queries.
6. Generate silhouettes from multiple viewpoints.
7. Capture screenshots.
8. Process data directly from memory.

### Expected output

You should obtain:

- A fully configured virtual acquisition setup;
- Interactive geometry exploration;
- A collection of silhouettes;
- Several screenshots documenting the workflow;
- A complete workflow without unnecessary intermediate files.

### Questions

- Which new feature provided the greatest productivity gain?
- Which feature is most useful for your research or teaching activities?
- How could these additions improve your existing gVXR workflows?